# Along-isobath averaging

Averaging a scalar along bathymetric contours rather than along
latitude circles, which is what Antarctic shelf analyses want
because the shelf break does not follow a latitude circle.

The original bins by bottom depth with `xhistogram`; `bin_sum`
below is the same operation without the xarray dependency.

Both of the original's caveats are properties of the method and
carry over unchanged:

1. deep troughs cutting into the shelf are averaged together with
   the open ocean at the same depth, and offshore seamounts with
   the slope. Choose the region mask with that in mind.
2. the bathymetry does not line up with the model's vertical
   levels, so the mask comes from `deptho` but the **binning
   coordinate is the depth of the deepest wet cell of the field
   itself**.

---

Translated from the COSIMA recipe
[`Along_Isobath_Average.ipynb`](https://github.com/COSIMA/cosima-recipes/blob/main/03-Advanced-Recipes/Along_Isobath_Average.ipynb)
onto CMORised CMIP6 data, using ESMValCore's `Dataset` and
preprocessors in place of the ACCESS-NRI intake catalog.

This notebook is standalone: it needs only `esmvalcore`,
`iris`, `numpy`, `matplotlib`, `cartopy` and `gsw`. The
equivalent ESMValTool recipe is
`recipes/recipe_cosima_*.yml`.

In [ ]:
import numpy as np
import iris
import iris.analysis.cartography
import iris.plot as iplt
import matplotlib.pyplot as plt

from esmvalcore.dataset import Dataset
from esmvalcore.preprocessor import (climate_statistics,
                                     extract_region, regrid)

## Start a dask cluster

Every COSIMA recipe opens with one, and these need one for the same
reason, though it is worth being precise about *which* part it speeds up.

**What the cluster does accelerate.** Everything from `Dataset.load()`
through the ESMValCore preprocessors — reading multi-decade files,
`regrid`, `climate_statistics`, `extract_region` — is lazy and chunked.
That is the expensive part of these notebooks and it is what the workers
parallelise. Without a client, dask falls back to its threaded scheduler;
ESMValCore itself warns that the distributed scheduler is preferable.

**What it does not.** Once the preprocessed cube is reduced, the
diagnostic arithmetic realises it into a plain masked array and proceeds
eagerly in numpy. That is deliberate: contour tracing, per-column
remapping and density binning are iterative and index-based, so they do
not express well as array graphs and would be slower, not faster, spread
over workers. Realising *after* the time-mean is what keeps that
affordable — realising a 20-year daily 3-D field before reducing it would
not be.

So: size the cluster for the load-and-preprocess stage. One thread per
worker, as the COSIMA recipes use — with chunks this large, per-worker
memory binds well before thread count does.

On ARE, `Client(threads_per_worker=1)` picks up the cores you requested
for the session. Pass `n_workers=` and `memory_limit=` to be explicit.

**Running the equivalent recipe instead?** Do not start a client there.
ESMValTool manages dask itself, and it defaults to the threaded
scheduler; switch it on in your ESMValTool configuration:

```yaml
dask:
  use: local_distributed
```

In [ ]:
import logging
import os

from dask.distributed import Client

# distributed logs every worker registration and teardown at INFO,
# which is hundreds of lines per notebook and buries everything the
# diagnostic prints. Warnings and errors still come through.
for _name in ("distributed", "distributed.scheduler", "distributed.nanny",
              "distributed.worker", "distributed.core",
              "distributed.batched", "distributed.utils_perf"):
    logging.getLogger(_name).setLevel(logging.WARNING)

# Client.current() reuses a cluster if this cell is re-run, instead of
# quietly starting a second one alongside the first.
try:
    worker_count = max(1, int(os.environ.get("PBS_NCPUS", "1")))
except ValueError:
    worker_count = 1
    print("PBS_NCPUS is not an integer; using one worker")
try:
    client = Client.current()
except ValueError:
    try:
        # Separate worker processes: real memory isolation, and what
        # you want on ARE or any batch node.
        client = Client(n_workers=worker_count, threads_per_worker=1)
    except (RuntimeError, OSError) as error:
        # Some containers and locked-down environments cannot spawn
        # worker subprocesses. In-process workers still give a
        # distributed scheduler and a dashboard, just no isolation.
        print(f'process workers unavailable ({error}); '
              f'using in-process workers instead')
        client = Client(n_workers=worker_count, threads_per_worker=1,
                        processes=False)

active_workers = len(client.scheduler_info()['workers'])
print('dashboard:', client.dashboard_link)
print(f'workers: {active_workers} active (requested {worker_count})')
client

## Load the data

In [ ]:
common = dict(project='CMIP6', exp='historical',
              ensemble='r1i1p1f1', grid='gn',
              dataset='ACCESS-CM2')

thetao = Dataset(short_name='thetao', mip='Omon',
                 timerange='2000/2009', **common)
so = thetao.copy(short_name='so')
deptho = Dataset(short_name='deptho', mip='Ofx', **common)

def ross_sea(cube):
    cube = regrid(cube, target_grid='0.5x0.5', scheme='linear')
    return extract_region(cube, start_longitude=160.,
                          end_longitude=210.,
                          start_latitude=-80., end_latitude=-68.)

thetao_cube = ross_sea(climate_statistics(
    thetao.load(), operator='mean', period='full'))
so_cube = ross_sea(climate_statistics(
    so.load(), operator='mean', period='full'))
deptho_cube = ross_sea(deptho.load())

## Helpers

In [ ]:
# The helpers below are lifted verbatim from
# cosima_cmip/cosima_common.py by notebooks/build_notebooks.py, so
# this notebook runs exactly the same code as the ESMValTool
# diagnostic script. Edit them there, then rebuild.
import contextlib
import warnings

try:  # TEOS-10, needed by the density helpers
    import gsw
except ImportError:
    gsw = None


EARTH_RADIUS = 6371000.0  # m, matches iris.analysis.cartography


FILL_VALUE_THRESHOLD = 1.0e19


def masked_data(cube, dtype=float):
    """Realised, fully masked data of a cube.

    Always go through this rather than ``cube.core_data()``.  The latter
    returns a *lazy* dask array, on which ``numpy.ma`` operations are
    silently no-ops: the mask is not applied, and fill values flow
    straight into the arithmetic.  This realises the data, applies any
    mask the file carries, and additionally masks non-finite values and
    unconverted fill values, which real CMOR files do contain.
    """
    data = np.ma.masked_invalid(np.ma.asarray(cube.data, dtype=dtype))
    return np.ma.masked_where(
        np.abs(np.ma.filled(data, 0.0)) >= FILL_VALUE_THRESHOLD, data)


def _coord_points(cube, name):
    """1D points for ``name``, taking column/row 0 of a 2D aux coord."""
    coord = cube.coord(name)
    if coord.ndim == 1:
        return coord.points
    points = coord.points
    # On a curvilinear grid latitude varies down a column and longitude
    # along a row; take the slice that actually varies.
    if name == "latitude":
        return points[:, 0]
    return points[0, :]


def lat_1d(cube):
    """Representative 1D latitude, also for 2D (curvilinear) coordinates."""
    return _coord_points(cube, "latitude")


def lon_1d(cube):
    """Representative 1D longitude, also for 2D coordinates."""
    return _coord_points(cube, "longitude")


def guess_bounds(cube, coords=("latitude", "longitude")):
    """Add bounds in place where they are missing."""
    for name in coords:
        if cube.coords(name) and not cube.coord(name).has_bounds():
            try:
                cube.coord(name).guess_bounds()
            except ValueError:  # scalar or single-point coordinate
                pass
    return cube


def depth_coord_name(cube):
    """Name of the vertical coordinate, or None for a 2D field."""
    for name in ("depth", "ocean_sigma_z", "olevel", "lev"):
        if cube.coords(name):
            return name
    for coord in cube.coords(dim_coords=True):
        if coord.units.is_convertible("m") and coord.name() != "latitude":
            return coord.name()
    return None


def surface_field(cube):
    """Shallowest level of a 3D field; a no-op for 2D fields."""
    name = depth_coord_name(cube)
    if name is None:
        return cube
    depth = cube.coord(name)
    index = int(np.argmin(np.abs(depth.points)))
    constraint = iris.Constraint(**{name: depth.points[index]})
    return cube.extract(constraint)


def horizontal_slice(cube):
    """First slice of a cube over its two horizontal dimensions.

    ``cube.slices(["latitude", "longitude"])`` cannot be used here: on a
    curvilinear grid the two coordinates share both dimensions and iris
    rejects them as non-orthogonal.  Slicing by dimension index works
    for regular and curvilinear grids alike.
    """
    horizontal_dims = sorted(
        set(cube.coord_dims("latitude")) | set(cube.coord_dims("longitude")))
    if cube.ndim == len(horizontal_dims):
        return cube
    index = tuple(slice(None) if dim in horizontal_dims else 0
                  for dim in range(cube.ndim))
    return cube[index]


def spherical_polygon_area(lon_vertices, lat_vertices, radius=EARTH_RADIUS):
    """Area in m2 of spherical polygons given their corner coordinates.

    ``lon_vertices`` and ``lat_vertices`` are in degrees with the corners
    along the last axis, which is the ``(..., 4)`` layout CMOR uses for
    ``vertices_longitude`` / ``vertices_latitude``.  Any number of
    corners works as long as they are ordered around the cell.

    The area is the spherical excess, accumulated over a triangle fan
    from the first corner, using the Van Oosterom & Strackee (1983)
    form:

        tan(E/2) = |v1 . (v2 x v3)|
                   / (1 + v1.v2 + v2.v3 + v3.v1)

    with ``v`` the unit vectors of the corners.  This is *exact* for
    great-circle edges and therefore invariant under rotation of the
    sphere, which matters because a tripolar grid is exactly a rotated
    and stretched one: a formula that is only correct for cells aligned
    with latitude circles would degrade precisely where the grid stops
    being aligned.

    The obvious alternative, the spherical shoelace formula
    ``R^2/2 sum (dlon)(sin lat_i + sin lat_{i+1})``, treats each edge as
    linear in ``(lon, sin lat)`` rather than as a great circle.  It is
    exact for a latitude-longitude box and accurate to second order for
    small cells, but it is not rotation invariant and it fails badly for
    cells spanning large longitude ranges, such as those beside a
    displaced pole.

    The two disagree by about 2e-4 relative on a 2-degree cell, and
    neither is "wrong": they assume different cell edges.  A cell whose
    north and south edges follow *parallels* has the shoelace area,
    which is what ``iris.analysis.cartography.area_weights`` returns; a
    cell with great-circle edges has this one.  Both tile the sphere
    exactly.  The discrepancy is far smaller than the difference
    between either and the model's own ``areacello``, which is why
    ``areacello`` is still preferred when it is published.
    """
    lon = np.deg2rad(np.asarray(lon_vertices, dtype=float))
    lat = np.deg2rad(np.asarray(lat_vertices, dtype=float))
    unit = np.stack([np.cos(lat) * np.cos(lon),
                     np.cos(lat) * np.sin(lon),
                     np.sin(lat)], axis=-1)

    first = unit[..., 0, :]
    excess = np.zeros(unit.shape[:-2])
    for corner in range(1, unit.shape[-2] - 1):
        second = unit[..., corner, :]
        third = unit[..., corner + 1, :]
        triple = np.abs(np.einsum("...i,...i->...",
                                  first, np.cross(second, third)))
        denominator = (1.0
                       + np.einsum("...i,...i->...", first, second)
                       + np.einsum("...i,...i->...", second, third)
                       + np.einsum("...i,...i->...", third, first))
        excess += 2.0 * np.arctan2(triple, denominator)
    return excess * radius**2


def area_from_bounds(cube, radius=EARTH_RADIUS):
    """Cell area in m2 from 2D coordinate bounds, or ``None``.

    This is what makes a tripolar grid workable without ``areacello``.
    CMOR requires bounds on ``latitude`` and ``longitude``, and on a
    curvilinear grid those bounds are the four corner vertices of every
    cell, so the area follows exactly from
    :func:`spherical_polygon_area`.

    ESMValCore does not do this: ``try_adding_calculated_cell_area``
    handles regular and rotated-pole grids and raises
    ``CoordinateMultiDimError`` for everything else.  The information is
    in the file regardless.

    Returns ``None`` when the cube has no usable 2D bounds, so callers
    can fall back.
    """
    horizontal = horizontal_slice(cube)
    lat_coord = horizontal.coord("latitude")
    lon_coord = horizontal.coord("longitude")
    if lat_coord.ndim != 2 or lon_coord.ndim != 2:
        return None
    if lat_coord.bounds is None or lon_coord.bounds is None:
        return None
    if lat_coord.bounds.shape[-1] < 3:
        return None
    return np.ma.masked_invalid(
        spherical_polygon_area(lon_coord.bounds, lat_coord.bounds,
                               radius=radius))


def cell_area(cube):
    """Cell area in m2, preferring the model's own ``areacello``.

    Order of preference:

    1. an ``areacello`` cell measure attached to the cube, which is what
       ESMValCore builds from a ``supplementary_variables`` entry.  This
       is the model's own area and is always the best answer: it
       accounts for the actual grid generation, including any partial
       cells at the coast.
    2. a spherical polygon area from the 2D corner vertices, via
       :func:`area_from_bounds`.  CMOR mandates bounds on latitude and
       longitude, so this works on a tripolar grid with no
       ``areacello`` at all.  It assumes great-circle cell edges on a
       sphere, so it differs from the model's own areas by well under a
       percent for an ocean grid.
    3. spherical areas from 1D latitude/longitude bounds, for a regular
       grid.

    Only if all three fail does this raise, and then with instructions.
    """
    horizontal = horizontal_slice(cube)

    for measure in horizontal.cell_measures():
        if measure.var_name in ("areacello", "areacella", "areacell"):
            return np.ma.masked_invalid(
                np.ma.asarray(
                    horizontal.cell_measure(measure.name()).core_data(),
                    dtype=float).reshape(horizontal.shape))

    from_bounds = area_from_bounds(horizontal)
    if from_bounds is not None:
        return from_bounds

    if horizontal.coord("latitude").ndim > 1:
        raise ValueError(
            "this cube has 2D (curvilinear) latitude/longitude and no cell "
            "corner bounds, so no cell area can be obtained. Either attach "
            "areacello -- in a recipe with\n"
            "    supplementary_variables:\n"
            "      - short_name: areacello\n"
            "        mip: Ofx\n"
            "(add 'exp: piControl' if the experiment you want does not "
            "publish it), in a notebook with "
            "dataset.add_supplementary(short_name='areacello', mip='Ofx') "
            "-- or keep the coordinate bounds, which CMOR requires and "
            "which are enough to compute the area exactly.")

    # iris defaults to an Earth radius of 6367470 m where the cube has
    # no coordinate system, while cell_lengths uses the mean radius of
    # 6371000 m. That is a 0.11% difference in area, small but free to
    # avoid: pin the same sphere so lengths and areas agree.
    horizontal = horizontal.copy()
    guess_bounds(horizontal)
    if horizontal.coord("latitude").coord_system is None:
        sphere = iris.coord_systems.GeogCS(EARTH_RADIUS)
        horizontal.coord("latitude").coord_system = sphere
        horizontal.coord("longitude").coord_system = sphere
    return iris.analysis.cartography.area_weights(horizontal)


def bottom_cell(cube):
    """Value in the deepest unmasked cell of every water column.

    ``Extract_Variables_at_Ocean_Bottom.ipynb`` uses MOM's ``kmt``;
    CMIP6 has no equivalent, so the bottom cell is found from the mask
    of the field itself.
    """
    name = depth_coord_name(cube)
    if name is None:
        return masked_data(cube)
    axis = cube.coord_dims(name)[0]
    data = masked_data(cube)
    data = np.ma.masked_array(np.moveaxis(data, axis, 0),
                              mask=np.moveaxis(
                                  np.ma.getmaskarray(data), axis, 0))
    valid = ~np.ma.getmaskarray(data)
    any_valid = valid.any(axis=0)
    # The deepest valid level is the last True along the depth axis.
    deepest = valid.shape[0] - 1 - np.argmax(valid[::-1], axis=0)
    result = np.take_along_axis(
        np.ma.filled(data, np.nan), deepest[None, ...], axis=0)[0]
    return np.ma.masked_array(result, mask=~any_valid)


def bin_centres(bins):
    """Midpoints of a monotonic array of bin edges."""
    bins = np.asarray(bins, dtype=float)
    return 0.5 * (bins[1:] + bins[:-1])


def bin_sum(values, bin_variable, bins, axis=None):
    """Sum ``values`` into ``bins`` of ``bin_variable``.

    ``values`` and ``bin_variable`` must have the same shape.  Returns
    an array whose first axis runs over the bin centres and whose
    remaining axes are the axes of ``values`` *not* listed in ``axis``.
    With ``axis=None`` everything is collapsed into a 1D profile.

    This is ``xhistogram.xarray.histogram(bin_variable, bins=bins,
    weights=values, dim=axis)`` without the xarray dependency.
    """
    values = np.ma.masked_invalid(np.ma.asarray(values, dtype=float))
    bin_variable = np.ma.masked_invalid(
        np.ma.asarray(bin_variable, dtype=float))
    bins = np.asarray(bins, dtype=float)
    n_bins = bins.size - 1

    if axis is None:
        axis = tuple(range(values.ndim))
    elif isinstance(axis, int):
        axis = (axis,)
    axis = tuple(a % values.ndim for a in axis)
    keep = tuple(a for a in range(values.ndim) if a not in axis)

    moved = np.ma.transpose(values, keep + axis)
    moved_bin = np.ma.transpose(bin_variable, keep + axis)
    kept_shape = moved.shape[:len(keep)]
    flat = moved.reshape(int(np.prod(kept_shape)) if keep else 1, -1)
    flat_bin = moved_bin.reshape(flat.shape)

    out = np.zeros((n_bins,) + kept_shape)
    flat_out = out.reshape(n_bins, -1)
    # np.digitize returns 0 below the first edge and n_bins+1 above the
    # last, so a bin index of 1..n_bins is inside the range.
    for column in range(flat.shape[0]):
        data = flat[column]
        coordinate = flat_bin[column]
        valid = ~np.ma.getmaskarray(data) & ~np.ma.getmaskarray(coordinate)
        if not valid.any():
            continue
        index = np.digitize(
            np.ma.filled(coordinate, np.nan)[valid], bins)
        inside = (index >= 1) & (index <= n_bins)
        flat_out[:, column] = np.bincount(
            index[inside] - 1,
            weights=np.ma.filled(data, 0.0)[valid][inside],
            minlength=n_bins)
    return out


@contextlib.contextmanager
def quiet_gsw():
    """Silence gsw's masked-array warning for the enclosed call.

    ``gsw`` emits ``'where' used without 'out', expect uninitialized
    memory in output`` every time it is handed a masked array, which is
    every time here.  The uninitialised values sit only underneath the
    mask and never reach the result -- asserted in
    ``validation/test_cosima_common.py`` -- but the warning fires once
    per call and buries everything else a notebook prints.

    Scoped to this one message so genuine gsw warnings still surface.
    """
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", message=".*'where' used without 'out'.*",
            category=UserWarning)
        yield


def _require_gsw():
    if gsw is None:
        raise ImportError(
            "gsw (TEOS-10) is required for the density diagnostics; "
            "install it with `pixi add gsw`")


def pressure_from_depth(depth, lat):
    """Sea pressure in dbar from depth in metres (positive down)."""
    _require_gsw()
    depth = np.asarray(depth, dtype=float)
    lat = np.asarray(lat, dtype=float)
    return gsw.p_from_z(-depth, lat)


def absolute_salinity(practical_salinity, pressure, lon, lat):
    """Practical salinity (PSU) to absolute salinity (g/kg)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.SA_from_SP(practical_salinity, pressure, lon, lat)


def conservative_temperature(abs_salinity, potential_temperature):
    """Potential temperature (degC) to conservative temperature (degC)."""
    _require_gsw()
    with quiet_gsw():
        return gsw.CT_from_pt(abs_salinity, potential_temperature)


def _broadcast_to_field(depth, lat, lon, shape):
    """Broadcast depth/lat/lon onto a ``(depth, lat, lon)`` field shape."""
    depth_b = np.broadcast_to(np.asarray(depth)[:, None, None], shape)
    lat_b = np.broadcast_to(np.asarray(lat)[None, :, None], shape)
    lon_b = np.broadcast_to(np.asarray(lon)[None, None, :], shape)
    return depth_b, lat_b, lon_b


def teos10_fields(thetao_cube, so_cube, temperature_is_conservative=False):
    """Absolute salinity, conservative temperature and pressure.

    Accepts fields with optional leading dimensions, such as time:
    ``(lat, lon)``, ``(depth, lat, lon)`` or
    ``(time, depth, lat, lon)``.  It returns masked arrays broadcast to
    the cube shape.  CMIP ``thetao`` is potential temperature, so it is
    converted unless the caller says otherwise (MOM5 ``temp`` is already
    conservative).
    """
    _require_gsw()
    salinity = masked_data(so_cube)
    temperature = masked_data(thetao_cube)

    if salinity.shape != temperature.shape:
        raise ValueError("thetao and so must have the same shape")

    def broadcast_1d(points, axis):
        """Broadcast a coordinate along its corresponding data axis."""
        reshape = [1] * salinity.ndim
        reshape[axis] = len(points)
        return np.broadcast_to(np.asarray(points).reshape(reshape),
                               salinity.shape)

    lat = lat_1d(so_cube)
    lon = lon_1d(so_cube)
    lat_axis = so_cube.coord_dims("latitude")[0]
    lon_axis = so_cube.coord_dims("longitude")[-1]
    depth_name = depth_coord_name(so_cube)
    if depth_name is None:
        depth_b = np.zeros(salinity.shape)
    else:
        depth = so_cube.coord(depth_name).points
        depth_axis = so_cube.coord_dims(depth_name)[0]
        depth_b = broadcast_1d(depth, depth_axis)
    lat_b = broadcast_1d(lat, lat_axis)
    lon_b = broadcast_1d(lon, lon_axis)
    pressure = pressure_from_depth(depth_b, lat_b)
    abs_sal = absolute_salinity(salinity, pressure, lon_b, lat_b)
    if temperature_is_conservative:
        cons_temp = temperature
    else:
        cons_temp = conservative_temperature(abs_sal, temperature)

    return abs_sal, cons_temp, pressure


def potential_density_anomaly(thetao_cube, so_cube, p_ref=2000,
                              temperature_is_conservative=False):
    """Potential density anomaly sigma_n in kg m-3.

    ``p_ref`` is the reference pressure in dbar; 0, 1000, 2000, 3000 and
    4000 map onto the ``gsw.sigma0``...``gsw.sigma4`` family.
    """
    _require_gsw()
    sigma_functions = {
        0: gsw.sigma0,
        1000: gsw.sigma1,
        2000: gsw.sigma2,
        3000: gsw.sigma3,
        4000: gsw.sigma4,
    }
    if p_ref not in sigma_functions:
        raise ValueError(
            f"p_ref must be one of {sorted(sigma_functions)}, got {p_ref}")
    abs_sal, cons_temp, _ = teos10_fields(
        thetao_cube, so_cube,
        temperature_is_conservative=temperature_is_conservative)
    with quiet_gsw():
        return np.ma.masked_invalid(
            sigma_functions[p_ref](abs_sal, cons_temp))

## Compute

In [ ]:
ISOBATH_BINS = np.arange(0.0, 4001.0, 100.0)
MASK_DEPTH = 2000.0

# Caveat 2: the binning coordinate is the bottom depth the 3D
# field itself sees, not the continuous deptho.
levels = thetao_cube.coord(depth_coord_name(thetao_cube)).points
marker = thetao_cube.copy(np.ma.masked_array(
    np.broadcast_to(levels[:, None, None],
                    thetao_cube.shape[-3:]).astype(float).copy(),
    mask=np.ma.getmaskarray(masked_data(thetao_cube))))
bottom_depth = bottom_cell(marker)

region_mask = masked_data(deptho_cube) < MASK_DEPTH

surface = surface_field(thetao_cube)
area = cell_area(surface)
binvar = np.ma.masked_where(~region_mask, bottom_depth)
area_in_bin = bin_sum(
    np.ma.masked_where(~region_mask, area), binvar,
    ISOBATH_BINS)
normalised_area = np.cumsum(area_in_bin)
normalised_area /= normalised_area[-1]

def isobath_section(cube):
    # Area-weighted 3-D scalar means by isobath and depth.  The
    # old generated notebook took only surface_field(cube); the
    # COSIMA recipe preserves every model level.
    data = masked_data(cube)
    section = np.ma.masked_all(
        (data.shape[0], len(ISOBATH_BINS) - 1))
    for level, field in enumerate(data):
        field = np.ma.masked_where(~region_mask, field)
        weights = np.ma.masked_where(
            np.ma.getmaskarray(field), area)
        weighted = bin_sum(field * weights, binvar,
                           ISOBATH_BINS)
        totals = bin_sum(weights, binvar, ISOBATH_BINS)
        with np.errstate(divide='ignore', invalid='ignore'):
            section[level] = np.ma.masked_invalid(
                np.where(totals > 0, weighted / totals, np.nan))
    return section

temperature_section = isobath_section(thetao_cube)
salinity_section = isobath_section(so_cube)
sigma2 = potential_density_anomaly(thetao_cube, so_cube,
                                   p_ref=2000)
sigma_cube = thetao_cube.copy(np.ma.filled(sigma2, np.nan))
density_section = isobath_section(sigma_cube)
depth_levels = thetao_cube.coord(
    depth_coord_name(thetao_cube)).points

# CMIP thetao is normally Kelvin; the upstream figure is Celsius.
if str(thetao_cube.units).lower() in ('k', 'kelvin'):
    temperature_section = temperature_section - 273.15

## Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5),
                         sharey=True)
panels = [
    (temperature_section, 'Temperature', '$^\circ$C',
     'RdYlBu_r'),
    (salinity_section, 'Salinity', str(so_cube.units),
     'viridis'),
    (density_section, r'$\sigma_2$', 'kg m$^{-3}$',
     'plasma'),
]
for ax, (section, title, unit, cmap) in zip(axes, panels):
    mesh = ax.pcolormesh(normalised_area, depth_levels, section,
                         cmap=cmap, shading='auto')
    if title != 'Temperature':
        density_lines = ax.contour(
            normalised_area, depth_levels, density_section,
            colors='0.35', linewidths=0.45, levels=5)
        ax.clabel(density_lines, fontsize=6, fmt='%.1f')
    ax.invert_yaxis()
    ax.set_xlabel('Normalised distance from Antarctica')
    ax.set_title(f'{title} along isobaths')
    fig.colorbar(mesh, ax=ax, shrink=0.82, label=unit)
axes[0].set_ylabel('Depth (m)')
fig.suptitle('Ross Sea isobath-averaged hydrography '
             f'(shallower than {MASK_DEPTH:.0f} m)')
plt.tight_layout()
fig

## Shut the cluster down

Workers hold their memory until the cluster closes. Leaving one running
in an idle kernel is the usual reason a later notebook cannot get the
memory it asks for.

In [ ]:
client.close()